# 09 — Series temporales

Trabajo con fechas, agrupaciones por período, ventanas móviles y métricas de cambio en el tiempo.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'
df = pd.read_csv(TRAIN, low_memory=False)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%d/%m/%Y')
print(df['Order Date'].dtype)
print(df['Order Date'].describe())


datetime64[us]
count                          9800
mean     2017-05-01 05:13:51.673469
min             2015-01-03 00:00:00
25%             2016-05-24 00:00:00
50%             2017-06-26 00:00:00
75%             2018-05-15 00:00:00
max             2018-12-30 00:00:00
Name: Order Date, dtype: object


## Accessor dt — extraer componentes de fecha

In [2]:
# Todos los atributos disponibles en .dt para columnas datetime
print(df['Order Date'].dt.year.value_counts().sort_index())
print()
print(df['Order Date'].dt.month.value_counts().sort_index())
print()
print('Día de semana (0=lunes):')
print(df['Order Date'].dt.dayofweek.value_counts().sort_index())


Order Date
2015    1953
2016    2055
2017    2534
2018    3258
Name: count, dtype: int64

Order Date
1      366
2      297
3      680
4      657
5      725
6      691
7      697
8      693
9     1354
10     809
11    1449
12    1382
Name: count, dtype: int64

Día de semana (0=lunes):
Order Date
0    1593
1    1889
2    1229
3     541
4    1067
5    1786
6    1695
Name: count, dtype: int64


In [3]:
# Duración entre fechas
df['dias_envio'] = (df['Ship Date'] - df['Order Date']).dt.days
print(df['dias_envio'].describe().round(1))
print()
print('Distribución días de envío:')
print(df['dias_envio'].value_counts().sort_index().head(10))


count    9800.0
mean        4.0
std         1.7
min         0.0
25%         3.0
50%         4.0
75%         5.0
max         7.0
Name: dias_envio, dtype: float64

Distribución días de envío:
dias_envio
0     514
1     363
2    1295
3     978
4    2718
5    2147
6    1170
7     615
Name: count, dtype: int64


## Agrupar por período con to_period()

`to_period('M')` convierte una fecha a su período mensual (2024-11). Permite agrupar por mes manteniendo el año — a diferencia de `dt.month` que agrupa todos los eneros juntos.

In [4]:
# Agrupar por mes conservando el año
ventas_mes = (
    df.groupby(df['Order Date'].dt.to_period('M'))['Sales']
    .sum()
    .reset_index()
    .rename(columns={'Order Date': 'periodo', 'Sales': 'revenue'})
)
ventas_mes['periodo'] = ventas_mes['periodo'].astype(str)
print(ventas_mes.tail(12))


    periodo      revenue
36  2018-01   43476.4740
37  2018-02   19920.9974
38  2018-03   58863.4128
39  2018-04   35541.9101
40  2018-05   43825.9822
41  2018-06   48190.7277
42  2018-07   44825.1040
43  2018-08   62837.8480
44  2018-09   86152.8880
45  2018-10   77448.1312
46  2018-11  117938.1550
47  2018-12   83030.3888


In [5]:
# Agrupar por trimestre
ventas_q = (
    df.groupby(df['Order Date'].dt.to_period('Q'))['Sales']
    .agg(revenue='sum', pedidos='count')
    .reset_index()
)
ventas_q['Order Date'] = ventas_q['Order Date'].astype(str)
print(ventas_q)


   Order Date      revenue  pedidos
0      2015Q1   73931.3960      277
1      2015Q2   85874.0936      382
2      2015Q3  142522.6063      555
3      2015Q4  177528.1122      739
4      2016Q1   62357.6870      249
5      2016Q2   87713.3730      431
6      2016Q3  128560.2072      579
7      2016Q4  180804.7382      796
8      2017Q1   92686.3650      333
9      2017Q2  135061.1610      585
10     2017Q3  138056.3742      720
11     2017Q4  234388.6498      896
12     2018Q1  122260.8842      484
13     2018Q2  127558.6200      675
14     2018Q3  193815.8400      890
15     2018Q4  278416.6750     1209


## resample() — agrupación temporal con DatetimeIndex

`resample` es más potente que `groupby(to_period)` para series temporales: permite frecuencias como `W` (semana), `BM` (fin de mes laboral), etc. Requiere que el índice sea `DatetimeIndex`.

In [6]:
# Establecer Order Date como índice para usar resample
df_idx = df.set_index('Order Date').sort_index()

# Agregar por mes
por_mes = df_idx['Sales'].resample('ME').sum()   # ME = Month End
print(por_mes.tail(12))
print()

# Agregar por semana
por_semana = df_idx['Sales'].resample('W').sum()
print(f'Semanas en el dataset: {len(por_semana)}')


Order Date
2018-01-31     43476.4740
2018-02-28     19920.9974
2018-03-31     58863.4128
2018-04-30     35541.9101
2018-05-31     43825.9822
2018-06-30     48190.7277
2018-07-31     44825.1040
2018-08-31     62837.8480
2018-09-30     86152.8880
2018-10-31     77448.1312
2018-11-30    117938.1550
2018-12-31     83030.3888
Freq: ME, Name: Sales, dtype: float64

Semanas en el dataset: 209


## rolling() — ventana móvil

Calcula estadísticas sobre una ventana de N períodos consecutivos. Útil para suavizar series ruidosas y detectar tendencias.

In [7]:
por_mes = df_idx['Sales'].resample('ME').sum().reset_index()
por_mes.columns = ['fecha', 'revenue']

# Media móvil de 3 meses
por_mes['media_movil_3m'] = por_mes['revenue'].rolling(window=3).mean().round(0)

# Los primeros N-1 valores son NaN porque no hay suficientes períodos anteriores
print(por_mes[['fecha', 'revenue', 'media_movil_3m']].tail(12))


        fecha      revenue  media_movil_3m
36 2018-01-31   43476.4740         72761.0
37 2018-02-28   19920.9974         53046.0
38 2018-03-31   58863.4128         40754.0
39 2018-04-30   35541.9101         38109.0
40 2018-05-31   43825.9822         46077.0
41 2018-06-30   48190.7277         42520.0
42 2018-07-31   44825.1040         45614.0
43 2018-08-31   62837.8480         51951.0
44 2018-09-30   86152.8880         64605.0
45 2018-10-31   77448.1312         75480.0
46 2018-11-30  117938.1550         93846.0
47 2018-12-31   83030.3888         92806.0


## pct_change(), diff() y shift()

In [8]:
# pct_change() — variación porcentual respecto al período anterior
por_mes['variacion_pct'] = por_mes['revenue'].pct_change().mul(100).round(1)

# diff() — diferencia absoluta respecto al período anterior
por_mes['variacion_abs'] = por_mes['revenue'].diff().round(0)

# shift(1) — desplaza la serie N posiciones (valor del período anterior)
por_mes['revenue_mes_anterior'] = por_mes['revenue'].shift(1)

print(por_mes[['fecha', 'revenue', 'variacion_pct', 'revenue_mes_anterior']].tail(8))


        fecha      revenue  variacion_pct  revenue_mes_anterior
40 2018-05-31   43825.9822           23.3            35541.9101
41 2018-06-30   48190.7277           10.0            43825.9822
42 2018-07-31   44825.1040           -7.0            48190.7277
43 2018-08-31   62837.8480           40.2            44825.1040
44 2018-09-30   86152.8880           37.1            62837.8480
45 2018-10-31   77448.1312          -10.1            86152.8880
46 2018-11-30  117938.1550           52.3            77448.1312
47 2018-12-31   83030.3888          -29.6           117938.1550


---
## Resumen

| Operación | Sintaxis |
|-----------|----------|
| Componentes de fecha | `df['col'].dt.year`, `.dt.month`, `.dt.dayofweek` |
| Duración | `(fecha2 - fecha1).dt.days` |
| Agrupar por mes+año | `df.groupby(df['col'].dt.to_period('M'))` |
| Resample por frecuencia | `df.set_index('fecha').resample('ME').sum()` |
| Ventana móvil | `.rolling(3).mean()` |
| Variación porcentual | `.pct_change()` |
| Diferencia absoluta | `.diff()` |
| Período anterior | `.shift(1)` |
